# LeetCode 4107 — Count Valid Suffix Sorts I (Medium)

Bài toán LeetCode:

```python
class Solution:
    def countSuffix(self, nums: List[int], target: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4107 (Weekly Contest 527 — câu 2).

Bài này dạy một tư duy rất mạnh: **tách một điều kiện phức tạp thành nhiều điều kiện đơn giản, độc lập**. Mỗi điều kiện đơn giản lại có dạng "đúng với mọi `i` nhỏ" hoặc "đúng với mọi `i` lớn", nên tập các `i` hợp lệ là **giao của hai khoảng** — đếm được trong `O(n)` mà không cần sắp xếp lần nào.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài: "sắp xếp hậu tố giảm dần" là gì, "không tăng" (non-increasing) nghĩa là gì.
2. Công cụ Python: `sorted(..., reverse=True)`, nối list, `collections.Counter` để so sánh **đa tập (multiset)**.
3. Brute Force `O(n² log n)` và vì sao quá chậm.
4. Cách tách điều kiện thành 3 phần: **tiền tố giữ nguyên**, **cùng đa tập**, **hậu tố của target không tăng**.
5. Kỹ thuật **mảng logic tiền tố / hậu tố** (`O(n)`).
6. Rút gọn thành **giao hai khoảng** `[minI, maxI]`, và mẹo "chỉ cần kiểm tra một ứng viên".
7. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai mảng số nguyên `nums` và `target`, cùng độ dài `n`.

Với mỗi chỉ số `i` (`0 <= i <= n - 1`), ta làm một thí nghiệm **độc lập** trên một bản sao của `nums`:

- giữ nguyên **tiền tố** `nums[0..i-1]`,
- **sắp xếp hậu tố** `nums[i..n-1]` theo thứ tự **không tăng** (từ lớn đến nhỏ, cho phép bằng nhau).

Đếm số chỉ số `i` mà mảng thu được **bằng đúng** `target`.

---

## Ví dụ 1

```python
nums = [1, 2, 3], target = [3, 2, 1]   ->   1
```

| i | tiền tố giữ nguyên | hậu tố sau khi sắp giảm dần | kết quả | bằng target? |
|---:|---|---|---|:---:|
| 0 | `[]` | `[3, 2, 1]` | `[3, 2, 1]` | ✔ |
| 1 | `[1]` | `[3, 2]` | `[1, 3, 2]` | |
| 2 | `[1, 2]` | `[3]` | `[1, 2, 3]` | |

## Ví dụ 2

```python
nums = [1, 5, 6, 4, 6], target = [1, 5, 6, 6, 4]   ->   2
```

- `i = 2`: giữ `[1, 5]`, sắp `[6, 4, 6]` → `[6, 6, 4]` → `[1, 5, 6, 6, 4]` ✔
- `i = 3`: giữ `[1, 5, 6]`, sắp `[4, 6]` → `[6, 4]` → `[1, 5, 6, 6, 4]` ✔
- Các `i` khác đều không ra `target`.

## Ví dụ 3

```python
nums = [7, 6, 8], target = [8, 6, 7]   ->   0
```

Sắp từ `i = 0, 1, 2` lần lượt cho `[8, 7, 6]`, `[7, 8, 6]`, `[7, 6, 8]` — không cái nào bằng `target`.

---

## Ràng buộc

```text
1 <= n == nums.length == target.length <= 10^5
1 <= nums[i], target[i] <= 10^5
```

# 2. Kiến thức Python nền tảng

## 2.1. Sắp xếp giảm dần và khái niệm "không tăng"

- `sorted(a)` sắp **tăng dần**; `sorted(a, reverse=True)` sắp **giảm dần**.
- Một dãy là **không tăng (non-increasing)** nếu mỗi phần tử `>=` phần tử ngay sau nó: `a[0] >= a[1] >= ... `. Cho phép bằng nhau, ví dụ `[6, 6, 4]` là không tăng.
- Sắp xếp giảm dần một dãy cho ra **dãy không tăng duy nhất** có cùng các phần tử.

In [1]:
a = [6, 4, 6]
print(sorted(a))                  # [4, 6, 6]
print(sorted(a, reverse=True))    # [6, 6, 4]


def is_non_increasing(arr) -> bool:
    return all(arr[k] >= arr[k + 1] for k in range(len(arr) - 1))


print(is_non_increasing([6, 6, 4]))   # True  (bằng nhau vẫn được)
print(is_non_increasing([5, 6, 4]))   # False (5 < 6 là một chỗ "đi lên")
print(is_non_increasing([7]))         # True  (một phần tử)

[4, 6, 6]
[6, 6, 4]
True
False
True


## 2.2. Ghép tiền tố và hậu tố

`nums[:i]` là tiền tố (các chỉ số `0..i-1`), `nums[i:]` là hậu tố (các chỉ số `i..n-1`). Dấu `+` nối hai list.

In [2]:
nums = [1, 5, 6, 4, 6]
i = 2
print("tiền tố:", nums[:i], "| hậu tố:", nums[i:])
print("kết quả thí nghiệm i = 2:", nums[:i] + sorted(nums[i:], reverse=True))

tiền tố: [1, 5] | hậu tố: [6, 4, 6]
kết quả thí nghiệm i = 2: [1, 5, 6, 6, 4]


## 2.3. Đa tập (multiset) và `collections.Counter`

Sắp xếp **không thêm, không bớt** phần tử nào — chỉ đổi chỗ. Vì vậy hai dãy "có thể sắp thành nhau" khi và chỉ khi chúng có **cùng các phần tử với cùng số lần xuất hiện**. Tập hợp có tính số lần lặp gọi là **đa tập (multiset)**.

`Counter(a)` đếm số lần xuất hiện của từng giá trị. Hai `Counter` so sánh bằng nhau khi mọi giá trị có cùng số đếm.

In [3]:
from collections import Counter

print(Counter([1, 5, 6, 4, 6]))
print(Counter([1, 5, 6, 4, 6]) == Counter([6, 4, 6, 5, 1]))   # True: chỉ khác thứ tự
print(Counter([1, 2]) == Counter([1, 1]))                     # False: khác số lần
# Cách khác (chậm hơn một chút): so sánh hai bản đã sắp xếp
print(sorted([1, 5, 6, 4, 6]) == sorted([6, 4, 6, 5, 1]))

Counter({6: 2, 1: 1, 5: 1, 4: 1})
True
False
True


## 2.4. Mảng logic tiền tố

Thay vì kiểm tra "tiền tố độ dài `i` của hai mảng có giống nhau không" cho từng `i` (mỗi lần `O(n)`), ta xây một lần mảng:

```text
pref_ok[i] = True  nếu nums[0..i-1] == target[0..i-1]
```

theo công thức truy hồi: `pref_ok[0] = True` (tiền tố rỗng luôn giống), `pref_ok[i + 1] = pref_ok[i] and nums[i] == target[i]`.

Một khi gặp vị trí khác nhau, mọi `pref_ok` phía sau đều `False`.

In [4]:
nums = [1, 5, 6, 4, 6]
target = [1, 5, 6, 6, 4]
n = len(nums)

pref_ok = [True] * (n + 1)
for i in range(n):
    pref_ok[i + 1] = pref_ok[i] and nums[i] == target[i]

for i in range(n + 1):
    print(f"pref_ok[{i}] = {pref_ok[i]!s:5}  (so sánh {nums[:i]} với {target[:i]})")

pref_ok[0] = True   (so sánh [] với [])
pref_ok[1] = True   (so sánh [1] với [1])
pref_ok[2] = True   (so sánh [1, 5] với [1, 5])
pref_ok[3] = True   (so sánh [1, 5, 6] với [1, 5, 6])
pref_ok[4] = False  (so sánh [1, 5, 6, 4] với [1, 5, 6, 6])
pref_ok[5] = False  (so sánh [1, 5, 6, 4, 6] với [1, 5, 6, 6, 4])


# 3. Cách 1 — Brute Force: làm đúng từng thí nghiệm

Với mỗi `i`, tạo mảng `nums[:i] + sorted(nums[i:], reverse=True)` rồi so với `target`.

In [5]:
from typing import List


def count_suffix_bruteforce(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        result = nums[:i] + sorted(nums[i:], reverse=True)
        if result == target:
            count += 1
    return count


print(count_suffix_bruteforce([1, 2, 3], [3, 2, 1]))               # 1
print(count_suffix_bruteforce([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]))   # 2
print(count_suffix_bruteforce([7, 6, 8], [8, 6, 7]))               # 0

1
2
0


## Độ phức tạp

Mỗi thí nghiệm sắp xếp một hậu tố dài `n - i` → `O(n log n)`, cộng thêm so sánh `O(n)`. Có `n` thí nghiệm → **`O(n² log n)`**.

In [6]:
import math

n = 10**5
print(f"Ước lượng số phép toán: {n * n * math.log2(n):,.0f}")

Ước lượng số phép toán: 166,096,404,744


Hơn một trăm tỷ phép toán — không thể. Ta cần trả lời "thí nghiệm `i` có thành công không" trong `O(1)` mà **không thật sự sắp xếp**.

---

# 4. Bước đột phá: tách điều kiện "kết quả == target" thành 3 phần

Kết quả của thí nghiệm `i` là `nums[:i] + sorted(nums[i:], reverse=True)`. Nó bằng `target` khi và chỉ khi **cả hai nửa** khớp:

**Nửa trái:** `nums[:i] == target[:i]`.

**Nửa phải:** `sorted(nums[i:], reverse=True) == target[i:]`. Điều này lại tương đương với **hai** điều kiện:

- `target[i:]` là dãy **không tăng** (vì kết quả sắp giảm dần luôn không tăng), **và**
- `nums[i:]` và `target[i:]` có **cùng đa tập** (vì sắp xếp chỉ đổi chỗ).

Ngược lại, nếu `target[i:]` không tăng và cùng đa tập với `nums[i:]`, thì `target[i:]` chính là **dãy không tăng duy nhất** tạo từ các phần tử đó, tức là `sorted(nums[i:], reverse=True)`.

Vậy thí nghiệm `i` thành công ⇔ cả 3 điều kiện:

| Ký hiệu | Điều kiện | Ý nghĩa |
|:---:|---|---|
| (A) | `nums[:i] == target[:i]` | tiền tố giữ nguyên phải khớp |
| (B) | `Counter(nums[i:]) == Counter(target[i:])` | hậu tố cùng các phần tử |
| (C) | `target[i:]` không tăng | hậu tố của target "trông như" đã sắp giảm dần |

## Rút gọn điều kiện (B)

Nếu (A) đúng (tiền tố giống hệt nhau), thì:

```text
cùng đa tập ở hậu tố   ⇔   cùng đa tập trên cả mảng
```

Vì đa tập của cả mảng = đa tập tiền tố + đa tập hậu tố, mà tiền tố đã giống nhau.

Điều kiện "cùng đa tập trên cả mảng" **không phụ thuộc `i`**! Ta chỉ cần kiểm tra **một lần** bằng `Counter(nums) == Counter(target)`. Nếu sai → đáp án là `0` ngay.

## Kiểm chứng trên ví dụ 2

`nums = [1, 5, 6, 4, 6]`, `target = [1, 5, 6, 6, 4]` — cùng đa tập `{1, 4, 5, 6, 6}` ✔

| i | (A) tiền tố khớp? | (C) `target[i:]` không tăng? | thành công? |
|---:|:---:|---|:---:|
| 0 | ✔ (rỗng) | `[1, 5, 6, 6, 4]` ✗ | |
| 1 | ✔ | `[5, 6, 6, 4]` ✗ | |
| 2 | ✔ | `[6, 6, 4]` ✔ | ✔ |
| 3 | ✔ | `[6, 4]` ✔ | ✔ |
| 4 | ✗ (`4 != 6` ở chỉ số 3) | `[4]` ✔ | |

Đáp án `2` ✔.

## Mảng logic hậu tố cho điều kiện (C)

Tương tự `pref_ok`, xây từ phải sang trái:

```text
suf_desc[i] = True nếu target[i..n-1] không tăng
suf_desc[n - 1] = True                         (một phần tử)
suf_desc[i] = suf_desc[i + 1] and target[i] >= target[i + 1]
```

In [7]:
target = [1, 5, 6, 6, 4]
n = len(target)

suf_desc = [True] * (n + 1)          # suf_desc[n] (hậu tố rỗng) cũng coi là True
for i in range(n - 2, -1, -1):
    suf_desc[i] = suf_desc[i + 1] and target[i] >= target[i + 1]

for i in range(n):
    print(f"suf_desc[{i}] = {suf_desc[i]!s:5}  (target[{i}:] = {target[i:]})")

suf_desc[0] = False  (target[0:] = [1, 5, 6, 6, 4])
suf_desc[1] = False  (target[1:] = [5, 6, 6, 4])
suf_desc[2] = True   (target[2:] = [6, 6, 4])
suf_desc[3] = True   (target[3:] = [6, 4])
suf_desc[4] = True   (target[4:] = [4])


# 5. Cách 2 — Mảng logic tiền tố / hậu tố: O(n)

1. Nếu `Counter(nums) != Counter(target)` → trả về `0`.
2. Xây `pref_ok` và `suf_desc`.
3. Đếm số `i` trong `0..n-1` có `pref_ok[i] and suf_desc[i]`.

In [8]:
def count_suffix_arrays(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):          # điều kiện (B), kiểm tra một lần
        return 0

    pref_ok = [True] * (n + 1)                    # điều kiện (A)
    for i in range(n):
        pref_ok[i + 1] = pref_ok[i] and nums[i] == target[i]

    suf_desc = [True] * (n + 1)                   # điều kiện (C)
    for i in range(n - 2, -1, -1):
        suf_desc[i] = suf_desc[i + 1] and target[i] >= target[i + 1]

    return sum(1 for i in range(n) if pref_ok[i] and suf_desc[i])


print(count_suffix_arrays([1, 2, 3], [3, 2, 1]))               # 1
print(count_suffix_arrays([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]))   # 2
print(count_suffix_arrays([7, 6, 8], [8, 6, 7]))               # 0

1
2
0


## Mô phỏng bằng `print`

In [9]:
def trace_arrays(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    same_multiset = Counter(nums) == Counter(target)
    print(f"nums = {nums}, target = {target}")
    print("Cùng đa tập?", same_multiset)
    if not same_multiset:
        return 0
    pref_ok = [True] * (n + 1)
    for i in range(n):
        pref_ok[i + 1] = pref_ok[i] and nums[i] == target[i]
    suf_desc = [True] * (n + 1)
    for i in range(n - 2, -1, -1):
        suf_desc[i] = suf_desc[i + 1] and target[i] >= target[i + 1]
    count = 0
    for i in range(n):
        ok = pref_ok[i] and suf_desc[i]
        count += ok
        print(f"i = {i}: (A) pref_ok = {pref_ok[i]!s:5} | (C) suf_desc = {suf_desc[i]!s:5} -> "
              f"{'THÀNH CÔNG' if ok else 'không'}")
    print("Đáp án:", count)
    return count


trace_arrays([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]);

nums = [1, 5, 6, 4, 6], target = [1, 5, 6, 6, 4]
Cùng đa tập? True
i = 0: (A) pref_ok = True  | (C) suf_desc = False -> không
i = 1: (A) pref_ok = True  | (C) suf_desc = False -> không
i = 2: (A) pref_ok = True  | (C) suf_desc = True  -> THÀNH CÔNG
i = 3: (A) pref_ok = True  | (C) suf_desc = True  -> THÀNH CÔNG
i = 4: (A) pref_ok = False | (C) suf_desc = True  -> không
Đáp án: 2


# 6. Bước đột phá 2: tập `i` hợp lệ là một KHOẢNG

Nhìn kỹ hai mảng logic:

- `pref_ok[i]` đúng với **mọi `i` nhỏ** rồi chuyển sang sai mãi mãi (khi gặp vị trí khác nhau đầu tiên). Gọi `maxI` là chỉ số khác nhau **đầu tiên** giữa `nums` và `target` → (A) đúng khi và chỉ khi `i <= maxI`. Nếu hai mảng giống hệt, (A) đúng với mọi `i`, đặt `maxI = n - 1`.
- `suf_desc[i]` thì ngược lại: sai với các `i` nhỏ, rồi đúng mãi mãi từ một điểm trở đi. Hậu tố `target[i:]` không tăng khi và chỉ khi nó **không chứa chỗ "đi lên"** nào. Gọi `minI` là chỉ số `j` **lớn nhất** có `target[j - 1] < target[j]` (chỗ đi lên cuối cùng) → (C) đúng khi và chỉ khi `i >= minI`. Nếu không có chỗ đi lên nào, `minI = 0`.

```text
i:          0   1   2   3   4
(A):        ✔   ✔   ✔   ✔   ✗        -> đúng khi i <= maxI = 3
(C):        ✗   ✗   ✔   ✔   ✔        -> đúng khi i >= minI = 2
cả hai:             ✔   ✔            -> khoảng [2, 3], có 2 phần tử
```

Vậy đáp án là độ dài của khoảng `[minI, maxI]`:

```text
answer = max(0, maxI - minI + 1)      (nếu cùng đa tập, ngược lại 0)
```

Vì sao `i >= minI` là đủ cho (C)? Chỗ đi lên tại `j` (`target[j-1] < target[j]`) nằm trong hậu tố `target[i:]` khi và chỉ khi `i <= j - 1`. Hậu tố không có chỗ đi lên nào ⇔ `i > j - 1` với mọi chỗ đi lên `j` ⇔ `i >= j` với `j` lớn nhất = `minI`.

# 7. Cách 3 — Giao hai khoảng: O(n), không cần mảng phụ (ngoài `Counter`)

In [10]:
def count_suffix_interval(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0

    max_i = n - 1                         # (A): i <= max_i
    for i in range(n):
        if nums[i] != target[i]:
            max_i = i                     # chỗ khác nhau đầu tiên
            break

    min_i = 0                             # (C): i >= min_i
    for j in range(n - 1, 0, -1):
        if target[j - 1] < target[j]:
            min_i = j                     # chỗ đi lên cuối cùng
            break

    return max(0, max_i - min_i + 1)


print(count_suffix_interval([1, 2, 3], [3, 2, 1]))               # 1
print(count_suffix_interval([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]))   # 2
print(count_suffix_interval([7, 6, 8], [8, 6, 7]))               # 0

1
2
0


## Mô phỏng ba ví dụ

| Ví dụ | Cùng đa tập? | maxI (khác nhau đầu tiên) | minI (đi lên cuối cùng) | Đáp án |
|---|:---:|---:|---:|---:|
| `[1,2,3]` vs `[3,2,1]` | ✔ | 0 | 0 (không có chỗ đi lên) | `0 - 0 + 1 = 1` |
| `[1,5,6,4,6]` vs `[1,5,6,6,4]` | ✔ | 3 | 2 (`5 < 6`) | `3 - 2 + 1 = 2` |
| `[7,6,8]` vs `[8,6,7]` | ✔ | 0 | 2 (`6 < 7`) | `max(0, 0 - 2 + 1) = 0` |

In [11]:
def trace_interval(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    print(f"nums = {nums}, target = {target}")
    if Counter(nums) != Counter(target):
        print("  khác đa tập -> 0")
        return 0
    max_i = next((i for i in range(n) if nums[i] != target[i]), n - 1)
    ascents = [j for j in range(1, n) if target[j - 1] < target[j]]
    min_i = ascents[-1] if ascents else 0
    print(f"  maxI = {max_i} (khác nhau đầu tiên; n - 1 nếu giống hệt)")
    print(f"  các chỗ đi lên trong target: {ascents} -> minI = {min_i}")
    answer = max(0, max_i - min_i + 1)
    print(f"  đáp án = max(0, {max_i} - {min_i} + 1) = {answer}")
    return answer


for a, b in [([1, 2, 3], [3, 2, 1]), ([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]), ([7, 6, 8], [8, 6, 7])]:
    trace_interval(a, b)

nums = [1, 2, 3], target = [3, 2, 1]
  maxI = 0 (khác nhau đầu tiên; n - 1 nếu giống hệt)
  các chỗ đi lên trong target: [] -> minI = 0
  đáp án = max(0, 0 - 0 + 1) = 1
nums = [1, 5, 6, 4, 6], target = [1, 5, 6, 6, 4]
  maxI = 3 (khác nhau đầu tiên; n - 1 nếu giống hệt)
  các chỗ đi lên trong target: [1, 2] -> minI = 2
  đáp án = max(0, 3 - 2 + 1) = 2
nums = [7, 6, 8], target = [8, 6, 7]
  maxI = 0 (khác nhau đầu tiên; n - 1 nếu giống hệt)
  các chỗ đi lên trong target: [2] -> minI = 2
  đáp án = max(0, 0 - 2 + 1) = 0


# 8. Cách 4 — "Chỉ cần thử một ứng viên"

Một góc nhìn sáng tạo khác, **không cần `Counter`**:

- Với mọi `i` trong khoảng `[minI, maxI]`, điều kiện (A) và (C) đã đúng.
- Điều kiện (B) tương đương "cùng đa tập trên cả mảng" — **giống nhau cho mọi `i` trong khoảng**.

Vậy hoặc **tất cả** `i` trong khoảng đều thành công, hoặc **không cái nào**. Chỉ cần làm **đúng một** thí nghiệm thật (ví dụ tại `i = maxI`) để biết!

Thí nghiệm tại `maxI` chỉ phải sắp xếp hậu tố `nums[maxI:]`, tốn `O(n log n)` trong trường hợp xấu — vẫn rất nhanh.

In [12]:
def count_suffix_one_check(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    max_i = next((i for i in range(n) if nums[i] != target[i]), n - 1)
    min_i = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] < target[j]:
            min_i = j
            break
    if min_i > max_i:
        return 0
    # thử thật một thí nghiệm tại i = max_i (tiền tố đã khớp, chỉ cần so phần hậu tố)
    if sorted(nums[max_i:], reverse=True) == target[max_i:]:
        return max_i - min_i + 1
    return 0


print(count_suffix_one_check([1, 2, 3], [3, 2, 1]))               # 1
print(count_suffix_one_check([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]))   # 2
print(count_suffix_one_check([7, 6, 8], [8, 6, 7]))               # 0
print(count_suffix_one_check([1, 2], [1, 1]))                     # 0: khác đa tập

1
2
0
0


# 9. Phiên bản đúng format LeetCode

Ta nộp cách 3 (giao hai khoảng).

In [13]:
from collections import Counter
from typing import List


class Solution:
    def countSuffix(self, nums: List[int], target: List[int]) -> int:
        n = len(nums)
        if Counter(nums) != Counter(target):
            return 0

        max_i = n - 1
        for i in range(n):
            if nums[i] != target[i]:
                max_i = i
                break

        min_i = 0
        for j in range(n - 1, 0, -1):
            if target[j - 1] < target[j]:
                min_i = j
                break

        return max(0, max_i - min_i + 1)

In [14]:
sol = Solution()

tests = [
    ([1, 2, 3], [3, 2, 1], 1, "ví dụ 1"),
    ([1, 5, 6, 4, 6], [1, 5, 6, 6, 4], 2, "ví dụ 2"),
    ([7, 6, 8], [8, 6, 7], 0, "ví dụ 3"),
    ([5], [5], 1, "một phần tử"),
    ([5], [6], 0, "một phần tử, khác"),
    ([3, 3, 3], [3, 3, 3], 3, "toàn bằng nhau: mọi i đều được"),
    ([3, 2, 1], [3, 2, 1], 3, "đã giảm dần sẵn"),
    ([1, 2, 3], [1, 2, 3], 1, "tăng dần, giống hệt: chỉ i = n - 1"),
    ([1, 2], [1, 1], 0, "khác đa tập"),
    ([2, 1, 3], [3, 2, 1], 1, "chỉ i = 0"),
]

for nums, target, expected, name in tests:
    result = sol.countSuffix(nums, target)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:36} -> {result}")

PASS | ví dụ 1                              -> 1
PASS | ví dụ 2                              -> 2
PASS | ví dụ 3                              -> 0
PASS | một phần tử                          -> 1
PASS | một phần tử, khác                    -> 0
PASS | toàn bằng nhau: mọi i đều được       -> 3
PASS | đã giảm dần sẵn                      -> 3
PASS | tăng dần, giống hệt: chỉ i = n - 1   -> 1
PASS | khác đa tập                          -> 0
PASS | chỉ i = 0                            -> 1


# 10. Vì sao đúng? (tóm tắt chứng minh)

1. **Tách điều kiện:** thí nghiệm `i` thành công ⇔ (A) tiền tố khớp, (B) hậu tố cùng đa tập, (C) `target[i:]` không tăng. Chiều "⇐" dựa trên sự thật: *mỗi đa tập có đúng một cách xếp không tăng*.
2. **Rút gọn (B):** khi (A) đúng, (B) ⇔ `Counter(nums) == Counter(target)` — không phụ thuộc `i`.
3. **(A) có dạng "tiền tố đóng":** nếu (A) đúng ở `i` thì đúng ở mọi `i' < i` (tiền tố ngắn hơn của hai tiền tố giống nhau vẫn giống nhau). Do đó (A) ⇔ `i <= maxI`.
4. **(C) có dạng "hậu tố đóng":** nếu `target[i:]` không tăng thì mọi hậu tố ngắn hơn cũng không tăng. Do đó (C) ⇔ `i >= minI`.
5. Giao của `{i <= maxI}` và `{i >= minI}` là khoảng `[minI, maxI]`, có `max(0, maxI - minI + 1)` phần tử.

---

## Trực giác

> Điều kiện "giữ nguyên tiền tố" muốn `i` **nhỏ** (tiền tố càng ngắn càng dễ khớp). Điều kiện "hậu tố của target phải trông như đã sắp giảm" muốn `i` **lớn** (hậu tố càng ngắn càng dễ không tăng). Hai lực ngược chiều này kẹp `i` vào một khoảng.

# 11. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | làm thật từng thí nghiệm | `O(n² log n)` | `O(n)` |
| 2. Mảng logic | `pref_ok`, `suf_desc` + `Counter` | `O(n)` | `O(n)` |
| 3. Giao hai khoảng | tìm `maxI`, `minI` + `Counter` | `O(n)` | `O(n)` cho `Counter` (tối đa số giá trị khác nhau) |
| 4. Một ứng viên | tìm `maxI`, `minI` + sắp một hậu tố | `O(n log n)` | `O(n)` |

Có thể thay `Counter` bằng một mảng đếm dài `10^5 + 1` (vì giá trị `<= 10^5`) — vẫn `O(n + V)`.

# 12. Đối chiếu các cách trên dữ liệu ngẫu nhiên

Để có nhiều bộ test "thành công", ta sinh `target` bằng cách lấy một thí nghiệm thật rồi đôi khi xáo trộn nhẹ.

In [15]:
import random

random.seed(4107)
nonzero = 0
for _ in range(4000):
    n = random.randint(1, 7)
    nums = [random.randint(1, 4) for _ in range(n)]
    if random.random() < 0.6:
        i = random.randint(0, n - 1)
        target = nums[:i] + sorted(nums[i:], reverse=True)
        if random.random() < 0.3:
            a, b = random.randrange(n), random.randrange(n)
            target[a], target[b] = target[b], target[a]
    else:
        target = [random.randint(1, 4) for _ in range(n)]

    expected = count_suffix_bruteforce(nums, target)
    assert count_suffix_arrays(nums, target) == expected, (nums, target)
    assert count_suffix_interval(nums, target) == expected, (nums, target)
    assert count_suffix_one_check(nums, target) == expected, (nums, target)
    assert sol.countSuffix(nums, target) == expected, (nums, target)
    nonzero += expected > 0

print(f"4000 bộ test ngẫu nhiên khớp nhau ({nonzero} bộ có đáp án > 0)")

4000 bộ test ngẫu nhiên khớp nhau (2275 bộ có đáp án > 0)


# 13. Những lỗi dễ mắc

## Lỗi 1 — Quên kiểm tra đa tập

Chỉ dùng (A) và (C) mà quên (B):

In [16]:
def wrong_no_multiset(nums, target):
    n = len(nums)
    max_i = next((i for i in range(n) if nums[i] != target[i]), n - 1)
    min_i = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] < target[j]:
            min_i = j
            break
    return max(0, max_i - min_i + 1)


print("Sai :", wrong_no_multiset([1, 2], [1, 1]))
print("Đúng:", sol.countSuffix([1, 2], [1, 1]))

Sai : 2
Đúng: 0


`target = [1, 1]` không tăng, tiền tố `[1]` khớp, nhưng `nums` có số `2` mà `target` không có → không thí nghiệm nào thành công.

## Lỗi 2 — Coi "bằng nhau" là chỗ đi lên

Dãy không tăng **cho phép bằng nhau**. Nếu viết `target[j - 1] <= target[j]` thì cặp `6, 6` ở ví dụ 2 bị coi là "đi lên" → `minI` sai.

In [17]:
def wrong_ascent_le(nums, target):
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0
    max_i = next((i for i in range(n) if nums[i] != target[i]), n - 1)
    min_i = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] <= target[j]:          # SAI: phải là <
            min_i = j
            break
    return max(0, max_i - min_i + 1)


print("Sai :", wrong_ascent_le([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]))
print("Đúng:", sol.countSuffix([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]))

Sai : 1
Đúng: 2


## Lỗi 3 — Đặt `maxI = n` khi hai mảng giống hệt

Chỉ số `i` chỉ chạy từ `0` tới `n - 1` (hậu tố không được rỗng theo đề). Nếu đặt `maxI = n` sẽ đếm thừa một thí nghiệm không tồn tại. Ví dụ `[3, 3, 3]` sẽ ra `4` thay vì `3`.

---

## Lỗi 4 — Lấy `maxI = (chỗ khác nhau đầu tiên) - 1`

Thí nghiệm `i` giữ nguyên tiền tố `0..i-1`, tức là **không** đụng tới chỉ số `i`. Nếu chỗ khác nhau đầu tiên là `d`, thí nghiệm `i = d` vẫn hợp lệ ở điều kiện (A) (tiền tố `0..d-1` khớp), và chỉ số `d` được "sửa" nhờ sắp xếp hậu tố. Vậy `maxI = d`, không phải `d - 1`.

---

## Lỗi 5 — Sắp xếp tăng dần

`sorted(nums[i:])` (không có `reverse=True`) là đề khác. Ví dụ 1 sẽ ra `0` thay vì `1`.

---

## Lỗi 6 — Brute Force cho `n = 10^5`

Đúng nhưng `O(n² log n)` → TLE.

# 14. Mẹo nhận diện

1. Đề yêu cầu **đếm số vị trí `i`** thỏa một điều kiện mà kiểm tra trực tiếp thì tốn kém (phải sắp xếp, phải dựng mảng mới...).
2. Hãy thử **tách điều kiện** thành các phần nhỏ, rồi hỏi với từng phần: *"nếu đúng ở `i` thì có đúng ở `i - 1` (hoặc `i + 1`) không?"*
   - Đúng ở mọi `i` nhỏ → điều kiện kiểu **tiền tố** → tìm ngưỡng trên.
   - Đúng ở mọi `i` lớn → điều kiện kiểu **hậu tố** → tìm ngưỡng dưới.
   - Không phụ thuộc `i` → kiểm tra **một lần**.
3. Kết quả là **giao các khoảng** → đếm bằng một phép trừ.
4. Khi cần kiểm tra từng `i` mà điều kiện không có dạng khoảng, dùng **mảng logic tiền tố / hậu tố** (`pref_ok`, `suf_ok`) xây một lần trong `O(n)`.
5. "Sắp xếp được thành nhau" ⇔ "cùng đa tập" → `Counter`.

Các bài cùng họ: *Shortest Unsorted Continuous Subarray* (LeetCode 581), *Next Permutation* (31), *Find All Good Indices* (2420), *Product of Array Except Self* (238).

# 15. Biến thể

## Biến thể 1 — Sắp hậu tố **tăng dần** (không giảm)

Chỉ cần đổi điều kiện (C): `target[i:]` phải **không giảm**, tức là không có chỗ "đi xuống" `target[j - 1] > target[j]`.

In [18]:
def count_suffix_ascending(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0
    max_i = next((i for i in range(n) if nums[i] != target[i]), n - 1)
    min_i = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] > target[j]:        # chỗ đi xuống cuối cùng
            min_i = j
            break
    return max(0, max_i - min_i + 1)


def brute_ascending(nums, target):
    return sum(nums[:i] + sorted(nums[i:]) == target for i in range(len(nums)))


print(count_suffix_ascending([3, 1, 2], [1, 2, 3]), brute_ascending([3, 1, 2], [1, 2, 3]))
print(count_suffix_ascending([1, 3, 2, 2], [1, 2, 2, 3]), brute_ascending([1, 3, 2, 2], [1, 2, 2, 3]))

1 1
2 2


## Biến thể 2 — Liệt kê các chỉ số hợp lệ

Vì tập hợp lệ là một khoảng, ta trả về `list(range(minI, maxI + 1))` (hoặc `[]`).

Lưu ý: cùng contest còn có bài **4109 — Count Valid Suffix Sorts II**, phiên bản khó hơn của bài này. Hiểu chắc tư duy "tách điều kiện + giao khoảng" ở đây là bước chuẩn bị tốt.

---

# 16. Template tư duy

```python
# Đếm số i thỏa (A) and (B) and (C)
if not condition_independent_of_i():       # ví dụ: cùng đa tập
    return 0
max_i = <ngưỡng trên của điều kiện kiểu tiền tố>
min_i = <ngưỡng dưới của điều kiện kiểu hậu tố>
return max(0, max_i - min_i + 1)
```

```python
# Khi không rút được thành khoảng: mảng logic
pref_ok = [True] * (n + 1)
for i in range(n):
    pref_ok[i + 1] = pref_ok[i] and <điều kiện tại i>
suf_ok = [True] * (n + 1)
for i in range(n - 1, -1, -1):
    suf_ok[i] = suf_ok[i + 1] and <điều kiện tại i>
```

# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Tính đáp án cho:

```python
nums = [2, 9, 4, 4, 7], target = [2, 9, 7, 4, 4]
```

Hãy xác định: cùng đa tập? `maxI`? các chỗ đi lên của `target` và `minI`? Kiểm tra bằng cell dưới.

In [19]:
# Kiểm tra Bài 1
trace_interval([2, 9, 4, 4, 7], [2, 9, 7, 4, 4]);

nums = [2, 9, 4, 4, 7], target = [2, 9, 7, 4, 4]
  maxI = 2 (khác nhau đầu tiên; n - 1 nếu giống hệt)
  các chỗ đi lên trong target: [1] -> minI = 1
  đáp án = max(0, 2 - 1 + 1) = 2


## Bài 2 — Dự đoán output

```python
print(sorted([3, 1, 3, 2], reverse=True))
print(Counter("abca") == Counter("caba"))
print([j for j in range(1, 5) if [4, 4, 5, 1, 2][j - 1] < [4, 4, 5, 1, 2][j]])
print(next((i for i in range(3) if [1, 2, 3][i] != [1, 2, 3][i]), "không có"))
```

In [20]:
print(sorted([3, 1, 3, 2], reverse=True))
print(Counter("abca") == Counter("caba"))
print([j for j in range(1, 5) if [4, 4, 5, 1, 2][j - 1] < [4, 4, 5, 1, 2][j]])
print(next((i for i in range(3) if [1, 2, 3][i] != [1, 2, 3][i]), "không có"))

[3, 3, 2, 1]
True
[2, 4]
không có


## Bài 3 — Hoàn thiện cách mảng logic

In [21]:
def exercise_3_arrays(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    # TODO 1: nếu khác đa tập thì return 0
    # TODO 2: xây pref_ok (độ dài n + 1)
    # TODO 3: xây suf_desc (độ dài n + 1), duyệt từ phải sang trái
    # TODO 4: đếm i trong 0..n-1 có pref_ok[i] and suf_desc[i]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_arrays([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]) == 2
# assert exercise_3_arrays([7, 6, 8], [8, 6, 7]) == 0

## Bài 4 — Tự viết cách giao hai khoảng

Không nhìn code phía trên. Yêu cầu: `O(n)`, xử lý đúng trường hợp hai mảng giống hệt.

In [22]:
def exercise_4_interval(nums: List[int], target: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_interval([3, 3, 3], [3, 3, 3]) == 3
# assert exercise_4_interval([1, 2, 3], [1, 2, 3]) == 1
# assert exercise_4_interval([1, 2], [1, 1]) == 0

## Bài 5 — Biến thể: sắp xếp **tiền tố**

Với mỗi `i` (`0 <= i <= n - 1`), sắp xếp **tiền tố** `nums[0..i]` theo thứ tự **tăng dần**, giữ nguyên phần còn lại. Đếm số `i` cho ra `target`.

```python
count_prefix_sorts([3, 1, 2, 5], [1, 2, 3, 5])   -> 2    # i = 2 và i = 3
count_prefix_sorts([2, 1], [1, 2])               -> 1    # chỉ i = 1
```

Gợi ý: điều kiện nào giờ là kiểu "tiền tố", điều kiện nào là kiểu "hậu tố"?

In [23]:
def exercise_5_count_prefix_sorts(nums: List[int], target: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_prefix_sorts([3, 1, 2, 5], [1, 2, 3, 5]) == 2
# assert exercise_5_count_prefix_sorts([2, 1], [1, 2]) == 1

## Bài 6 — Mini challenge: được chọn sắp tăng HOẶC giảm

Với mỗi `i`, ta được chọn sắp hậu tố `nums[i:]` **tăng dần hoặc giảm dần** (chọn một). Đếm số `i` mà **ít nhất một** lựa chọn cho ra `target`.

```python
count_either([1, 3, 2], [1, 2, 3])       -> 2    # i = 0 và i = 1 (đều nhờ sắp tăng)
count_either([5, 5, 5], [5, 5, 5])       -> 3
count_either([2, 1, 3], [3, 1, 2])       -> 0
```

Gợi ý: tập `i` hợp lệ cho mỗi lựa chọn là một khoảng; đáp án là kích thước **hợp** của hai khoảng.

In [24]:
def exercise_6_count_either(nums: List[int], target: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_count_either([1, 3, 2], [1, 2, 3]) == 2
# assert exercise_6_count_either([5, 5, 5], [5, 5, 5]) == 3
# assert exercise_6_count_either([2, 1, 3], [3, 1, 2]) == 0

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `pref_ok[0] = True`, `suf_desc[n - 1] = True`.
- Mức 2: `pref_ok[i + 1] = pref_ok[i] and nums[i] == target[i]`.
- Mức 3: `suf_desc[i] = suf_desc[i + 1] and target[i] >= target[i + 1]` với `i` từ `n - 2` về `0`.

**Bài 4**

- Mức 1: `maxI` = chỗ khác nhau đầu tiên, mặc định `n - 1`.
- Mức 2: `minI` = chỗ đi lên cuối cùng (`target[j-1] < target[j]`), mặc định `0`.
- Mức 3: `max(0, maxI - minI + 1)` sau khi đã kiểm tra `Counter`.

**Bài 5**

- Mức 1: phần **giữ nguyên** bây giờ là hậu tố `nums[i+1:]` → phải khớp `target[i+1:]` → điều kiện kiểu "hậu tố" (đúng với `i` lớn).
- Mức 2: phần được sắp `target[0..i]` phải **không giảm** → điều kiện kiểu "tiền tố" (đúng với `i` nhỏ). Ngưỡng là chỗ "đi xuống" đầu tiên.
- Mức 3: `minI` = chỗ khác nhau **cuối cùng** (mặc định `0`); `maxI` = (chỗ đi xuống đầu tiên `j` có `target[j] < target[j-1]`) `- 1`, mặc định `n - 1`. Nhớ kiểm tra `Counter`.

**Bài 6**

- Mức 1: tính khoảng `[lo1, hi1]` cho sắp giảm và `[lo2, hi2]` cho sắp tăng (`maxI` giống nhau!).
- Mức 2: hai khoảng có chung đầu mút phải `maxI`, nên hợp của chúng là khoảng `[min(lo1, lo2), maxI]`.
- Mức 3: đáp án `max(0, maxI - min(lo1, lo2) + 1)` nếu cùng đa tập.

# 18. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

- Đa tập: `{2, 4, 4, 7, 9}` ở cả hai → ✔
- `maxI`: so sánh `2=2, 9=9, 4≠7` → `maxI = 2`.
- `target = [2, 9, 7, 4, 4]`: chỗ đi lên chỉ có `2 < 9` tại `j = 1` → `minI = 1`.
- Đáp án: `2 - 1 + 1 = 2` (các thí nghiệm `i = 1` và `i = 2`).

## Đáp án Bài 2

```text
[3, 3, 2, 1]
True
[2, 4]
không có
```

In [25]:
# Đáp án Bài 3
def answer_3_arrays(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0
    pref_ok = [True] * (n + 1)
    for i in range(n):
        pref_ok[i + 1] = pref_ok[i] and nums[i] == target[i]
    suf_desc = [True] * (n + 1)
    for i in range(n - 2, -1, -1):
        suf_desc[i] = suf_desc[i + 1] and target[i] >= target[i + 1]
    return sum(1 for i in range(n) if pref_ok[i] and suf_desc[i])


assert answer_3_arrays([1, 5, 6, 4, 6], [1, 5, 6, 6, 4]) == 2
assert answer_3_arrays([7, 6, 8], [8, 6, 7]) == 0
assert answer_3_arrays([2, 9, 4, 4, 7], [2, 9, 7, 4, 4]) == 2
print("Bài 3: PASS")

Bài 3: PASS


In [26]:
# Đáp án Bài 4
def answer_4_interval(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0
    max_i = n - 1
    for i in range(n):
        if nums[i] != target[i]:
            max_i = i
            break
    min_i = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] < target[j]:
            min_i = j
            break
    return max(0, max_i - min_i + 1)


assert answer_4_interval([3, 3, 3], [3, 3, 3]) == 3
assert answer_4_interval([1, 2, 3], [1, 2, 3]) == 1
assert answer_4_interval([1, 2], [1, 1]) == 0
print("Bài 4: PASS")

Bài 4: PASS


In [27]:
# Đáp án Bài 5
def answer_5_count_prefix_sorts(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0
    min_i = 0                                  # hậu tố nums[i+1:] phải khớp -> i >= chỗ khác nhau cuối cùng
    for i in range(n - 1, -1, -1):
        if nums[i] != target[i]:
            min_i = i
            break
    max_i = n - 1                              # target[0..i] không giảm -> i < chỗ đi xuống đầu tiên
    for j in range(1, n):
        if target[j] < target[j - 1]:
            max_i = j - 1
            break
    return max(0, max_i - min_i + 1)


def brute_prefix_sorts(nums, target):
    return sum(sorted(nums[:i + 1]) + nums[i + 1:] == target for i in range(len(nums)))


assert answer_5_count_prefix_sorts([3, 1, 2, 5], [1, 2, 3, 5]) == 2
assert answer_5_count_prefix_sorts([2, 1], [1, 2]) == 1
random.seed(5)
for _ in range(2000):
    n = random.randint(1, 6)
    a = [random.randint(1, 3) for _ in range(n)]
    i = random.randint(0, n - 1)
    b = sorted(a[:i + 1]) + a[i + 1:] if random.random() < 0.7 else [random.randint(1, 3) for _ in range(n)]
    assert answer_5_count_prefix_sorts(a, b) == brute_prefix_sorts(a, b), (a, b)
print("Bài 5: PASS (kể cả 2000 test ngẫu nhiên)")

Bài 5: PASS (kể cả 2000 test ngẫu nhiên)


In [28]:
# Đáp án Bài 6
def answer_6_count_either(nums: List[int], target: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(target):
        return 0
    max_i = next((i for i in range(n) if nums[i] != target[i]), n - 1)
    lo_desc = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] < target[j]:
            lo_desc = j
            break
    lo_asc = 0
    for j in range(n - 1, 0, -1):
        if target[j - 1] > target[j]:
            lo_asc = j
            break
    return max(0, max_i - min(lo_desc, lo_asc) + 1)


def brute_either(nums, target):
    return sum(
        nums[:i] + sorted(nums[i:]) == target or nums[:i] + sorted(nums[i:], reverse=True) == target
        for i in range(len(nums))
    )


assert answer_6_count_either([1, 3, 2], [1, 2, 3]) == 2
assert answer_6_count_either([5, 5, 5], [5, 5, 5]) == 3
assert answer_6_count_either([2, 1, 3], [3, 1, 2]) == 0
random.seed(6)
for _ in range(2000):
    n = random.randint(1, 6)
    a = [random.randint(1, 3) for _ in range(n)]
    i = random.randint(0, n - 1)
    b = a[:i] + sorted(a[i:], reverse=random.random() < 0.5)
    assert answer_6_count_either(a, b) == brute_either(a, b), (a, b)
print("Bài 6: PASS (kể cả 2000 test ngẫu nhiên)")

Bài 6: PASS (kể cả 2000 test ngẫu nhiên)


# 19. Tổng kết

Bài **Count Valid Suffix Sorts I** dạy cách **đếm mà không mô phỏng**:

1. Tách "kết quả == target" thành (A) tiền tố khớp, (B) cùng đa tập, (C) `target[i:]` không tăng.
2. (B) không phụ thuộc `i` → kiểm tra một lần bằng `Counter`.
3. (A) là điều kiện kiểu tiền tố → `i <= maxI` (chỗ khác nhau đầu tiên). (C) là điều kiện kiểu hậu tố → `i >= minI` (chỗ đi lên cuối cùng).
4. Đáp án = độ dài khoảng `[minI, maxI]`.

## Công thức nhớ nhanh

```text
answer = 0                               nếu Counter(nums) != Counter(target)
answer = max(0, maxI - minI + 1)         ngược lại
maxI = chỗ khác nhau đầu tiên (n - 1 nếu giống hệt)
minI = chỗ đi lên cuối cùng của target (0 nếu không có)
```

## Phiên bản nên nhớ

```python
class Solution:
    def countSuffix(self, nums: List[int], target: List[int]) -> int:
        n = len(nums)
        if Counter(nums) != Counter(target):
            return 0
        max_i = n - 1
        for i in range(n):
            if nums[i] != target[i]:
                max_i = i
                break
        min_i = 0
        for j in range(n - 1, 0, -1):
            if target[j - 1] < target[j]:
                min_i = j
                break
        return max(0, max_i - min_i + 1)
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)   (Counter)
```

# 20. Checklist trước khi submit

- [ ] Đã kiểm tra **cùng đa tập** (`Counter`) trước tiên?
- [ ] `maxI` là chỗ khác nhau **đầu tiên** (không trừ 1), mặc định `n - 1` (không phải `n`)?
- [ ] `minI` là chỗ **đi lên** cuối cùng với điều kiện **`<`** nghiêm ngặt (bằng nhau không tính)?
- [ ] Trả về `max(0, ...)` để không ra số âm khi khoảng rỗng?
- [ ] Sắp xếp **giảm dần** (không tăng), không nhầm sang tăng dần?
- [ ] Đã test: một phần tử, mảng toàn bằng nhau, hai mảng giống hệt, khác đa tập?